# Large-Scale Fraud Detection — EDA & Model Training

This notebook builds the offline ML side of the system. It deliberately keeps the workflow inside the notebook so the repository remains easy to understand during an interview.

**Dataset:** public `Fraud_Data.csv` from `rashida048/Datasets`.


In [1]:
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
import joblib

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data'
MODEL_DIR = ROOT / 'models'
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)


In [2]:
DATA_URL = 'https://raw.githubusercontent.com/rashida048/Datasets/master/fraud_data.csv'
data_path = DATA_DIR / 'fraud_data.csv'

if not data_path.exists():
    response = requests.get(DATA_URL, timeout=60)
    response.raise_for_status()
    data_path.write_bytes(response.content)

df = pd.read_csv(data_path)
print('Shape:', df.shape)
df.head()


Shape: (21693, 30)


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,1.176563,0.323798,0.536927,1.047002,-0.368652,-0.728586,0.084678,-0.069246,-0.266389,0.155315,...,-0.109627,-0.341365,0.057845,0.499180,0.415211,-0.581949,0.015472,0.018065,4.67,0
1,0.681109,-3.934776,-3.801827,-1.147468,-0.735540,-0.501097,1.038865,-0.626979,-2.274423,1.527782,...,0.652202,0.272684,-0.982151,0.165900,0.360251,0.195321,-0.256273,0.056501,912.00,0
2,1.140729,0.453484,0.247010,2.383132,0.343287,0.432804,0.093380,0.173310,-0.808999,0.775436,...,-0.003802,0.058556,-0.121177,-0.304215,0.645893,0.122600,-0.012115,-0.005945,1.00,0
3,-1.107073,-3.298902,-0.184092,-1.795744,2.137564,-1.684992,-2.015606,-0.007181,-0.165760,0.869659,...,0.130648,0.329445,0.927656,-0.049560,-1.892866,-0.575431,0.266573,0.414184,62.10,0
4,-0.314818,0.866839,-0.124577,-0.627638,2.651762,3.428128,0.194637,0.670674,-0.442658,0.133499,...,-0.312774,-0.799494,-0.064488,0.953062,-0.429550,0.158225,0.076943,-0.015051,2.67,0


## 1. Basic data checks

The first checks should be simple: shape, missing values, duplicate records and target distribution. Fraud datasets are usually highly imbalanced, so accuracy alone is not a useful operational metric.


In [4]:
print(df.info())
display(df.isna().sum().sort_values(ascending=False).head(10))
print('Duplicates:', df.duplicated().sum())
print(df['Class'].value_counts(normalize=True))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21693 entries, 0 to 21692
Data columns (total 30 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   V1      21693 non-null  float64
 1   V2      21693 non-null  float64
 2   V3      21693 non-null  float64
 3   V4      21693 non-null  float64
 4   V5      21693 non-null  float64
 5   V6      21693 non-null  float64
 6   V7      21693 non-null  float64
 7   V8      21693 non-null  float64
 8   V9      21693 non-null  float64
 9   V10     21693 non-null  float64
 10  V11     21693 non-null  float64
 11  V12     21693 non-null  float64
 12  V13     21693 non-null  float64
 13  V14     21693 non-null  float64
 14  V15     21693 non-null  float64
 15  V16     21693 non-null  float64
 16  V17     21693 non-null  float64
 17  V18     21693 non-null  float64
 18  V19     21693 non-null  float64
 19  V20     21693 non-null  float64
 20  V21     21693 non-null  float64
 21  V22     21693 non-null  float64
 22

V1     0
V2     0
V3     0
V4     0
V5     0
V6     0
V7     0
V8     0
V9     0
V10    0
dtype: int64

Duplicates: 143
Class
0    0.983589
1    0.016411
Name: proportion, dtype: float64


## 2. Feature engineering

We turn timestamps into hour/day features and create transaction behaviour features. In production, the same feature definitions must be available both offline and online. Notebook 02 demonstrates the online side.


In [9]:
# Build a feature matrix compatible with the real public fraud dataset schema.
df = df.copy()

# Optional timestamp conversion only if those columns exist.
for col in ['purchase_time', 'signup_time']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Create time-based features only when the relevant columns exist.
if {'purchase_time', 'signup_time'}.issubset(df.columns):
    df['hour'] = df['purchase_time'].dt.hour
    df['dayofweek'] = df['purchase_time'].dt.dayofweek
    df['account_age_days'] = (
        (df['purchase_time'] - df['signup_time']).dt.total_seconds() / 86400.0
    )
else:
    df['hour'] = 0
    df['dayofweek'] = 0
    df['account_age_days'] = 0.0

# Create amount-based features only if Amount exists.
if 'Amount' in df.columns:
    df['amount_sq'] = df['Amount'] ** 2
    df['abs_amount'] = df['Amount'].abs()
    df['amount_rank'] = df['Amount'].rank(pct=True)

# Keep all numeric features except the target.
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != 'Class']
features = [c for c in numeric_cols if c not in {'hour', 'dayofweek', 'account_age_days'}]

for col in ['hour', 'dayofweek', 'account_age_days', 'amount_sq', 'abs_amount', 'amount_rank']:
    if col in df.columns:
        features.append(col)

features = list(dict.fromkeys(features))

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df['Class'].astype(int)

X.head()


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V26,V27,V28,Amount,amount_sq,abs_amount,amount_rank,hour,dayofweek,account_age_days
0,1.176563,0.323798,0.536927,1.047002,-0.368652,-0.728586,0.084678,-0.069246,-0.266389,0.155315,...,-0.581949,0.015472,0.018065,4.67,21.8089,4.67,0.231826,0,0,0.0
1,0.681109,-3.934776,-3.801827,-1.147468,-0.735540,-0.501097,1.038865,-0.626979,-2.274423,1.527782,...,0.195321,-0.256273,0.056501,912.00,831744.0000,912.00,0.988107,0,0,0.0
2,1.140729,0.453484,0.247010,2.383132,0.343287,0.432804,0.093380,0.173310,-0.808999,0.775436,...,0.122600,-0.012115,-0.005945,1.00,1.0000,1.00,0.086387,0,0,0.0
3,-1.107073,-3.298902,-0.184092,-1.795744,2.137564,-1.684992,-2.015606,-0.007181,-0.165760,0.869659,...,-0.575431,0.266573,0.414184,62.10,3856.4100,62.10,0.710667,0,0,0.0
4,-0.314818,0.866839,-0.124577,-0.627638,2.651762,3.428128,0.194637,0.670674,-0.442658,0.133499,...,0.158225,0.076943,-0.015051,2.67,7.1289,2.67,0.185659,0,0,0.0


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = HistGradientBoostingClassifier(
    max_iter=160, learning_rate=0.06, max_leaf_nodes=20, random_state=42
)
model.fit(X_train, y_train)

prob = model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.50).astype(int)
print(classification_report(y_test, pred, digits=4))
print('ROC-AUC:', round(roc_auc_score(y_test, prob), 4))
print('PR-AUC:', round(average_precision_score(y_test, prob), 4))


              precision    recall  f1-score   support

           0     0.9981    0.9995    0.9988      4268
           1     0.9692    0.8873    0.9265        71

    accuracy                         0.9977      4339
   macro avg     0.9837    0.9434    0.9626      4339
weighted avg     0.9977    0.9977    0.9976      4339

ROC-AUC: 0.991
PR-AUC: 0.9456


## 3. Save the model bundle

The Flask application uses the same feature names. Saving the feature list with the model prevents accidental column-order mismatches at inference time.


In [11]:
joblib.dump({'model': model, 'features': features}, MODEL_DIR / 'fraud_model.joblib')
print('Saved:', MODEL_DIR / 'fraud_model.joblib')


Saved: c:\Users\user\Desktop\Large-Scale-Fraud-Detection\models\fraud_model.joblib


## 4. Interview discussion points

- Why is recall important for fraud detection?
- Why can a model with high accuracy still be operationally poor?
- Which features must be calculated without looking into the future?
- How would you prevent training/serving skew?
- How would Kafka and an online feature store change the architecture?
- How would you monitor drift and false-positive rates after deployment?
